In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Pantau atau batalkan kerja

Lihat senarai beban kerja anda di [halaman Beban Kerja](https://quantum.cloud.ibm.com/workloads).

## Lihat status kerja
Pergi ke [jadual Beban Kerja](https://quantum.cloud.ibm.com/workloads) anda dan semak di bawah lajur Status sama ada kerja telah selesai atau gagal.

## Lihat baki penggunaan
Pergi ke [jadual Instances](https://quantum.cloud.ibm.com/instances) anda dan pilih tab yang berkaitan dengan pelan yang anda ingin lihat baki penggunaannya. Jumlah masa yang digunakan dan jumlah masa yang tinggal pada pelan anda akan dipaparkan.

## Lihat metrik bilangan kerja dan beban kerja yang diserahkan
Pergi ke [halaman Analitik](https://quantum.cloud.ibm.com/analytics) untuk melihat jumlah bilangan kerja yang diserahkan, serta kiraan beban kerja kumpulan dan beban kerja Session. Ambil perhatian bahawa anda hanya boleh melihat halaman Analitik untuk akaun yang anda miliki atau uruskan.

## Pantau kerja
Gunakan instance kerja untuk menyemak status kerja atau mendapatkan semula keputusan dengan memanggil arahan yang sesuai:

|                               |                                                                                                                                                                                                              |
| ----------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| job.result()                  | Semak keputusan kerja serta-merta selepas kerja selesai. Keputusan kerja tersedia selepas kerja selesai. Oleh itu, job.result() adalah panggilan penyekatan sehingga kerja selesai.                               |
| job.job\_id()                 | Kembalikan ID yang mengenal pasti kerja itu secara unik. Mendapatkan semula keputusan kerja pada masa kemudian memerlukan ID kerja. Oleh itu, adalah disyorkan agar anda menyimpan ID kerja yang mungkin ingin anda dapatkan semula kemudian. |
| job.status()                  | Semak status kerja.                                                                                                                                                                                        |
| job = service.job(\<job\_id>) | Dapatkan semula kerja yang anda serahkan sebelum ini. Panggilan ini memerlukan ID kerja.                                                                                                                                      |

<span id="retrieve-later"></span>
## Dapatkan semula keputusan kerja pada masa kemudian
Panggil `service.job(\<job\_id>)` untuk mendapatkan semula kerja yang anda serahkan sebelum ini. Jika anda tidak mempunyai ID kerja, atau jika anda ingin mendapatkan semula berbilang kerja sekaligus; termasuk kerja dari QPU (unit pemprosesan kuantum) yang telah bersara, panggil `service.jobs()` dengan penapis pilihan sebaliknya. Lihat [QiskitRuntimeService.jobs](../api/qiskit-ibm-runtime/qiskit-runtime-service#jobs).

> **Note:** `service.jobs()` juga mengembalikan kerja yang dijalankan dari pakej `qiskit-ibm-provider` yang telah ditamatkan. Kerja yang diserahkan oleh pakej `qiskit-ibmq-provider` yang lebih lama (juga telah ditamatkan) tidak lagi tersedia.

### Contoh
Contoh ini mengembalikan 10 kerja runtime terkini yang dijalankan pada `my_backend`:

In [ ]:
# This cell is hidden from users
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
import numpy as np


my_backend = "ibm_torino"
service = QiskitRuntimeService()
# backend = service.backend(my_backend)
backend = service.least_busy()

# Define two circuits, each with one parameter with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure_all()


pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)

params = np.random.uniform(size=(2, 3)).T

sampler_pub = (transpiled_circuit, params)

# Instantiate the new Estimator object, then run the transpiled circuit
# using the set of parameters and observables.
sampler = SamplerV2(mode=backend)
job = sampler.run([sampler_pub], shots=4)
print(job.job_id())

d305ck0ocacs73ajagvg


In [2]:
result = job.result()


spans = job.result().metadata["execution"]["execution_spans"]
print(spans)

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])


In [3]:
params = np.random.uniform(size=(2, 3))
params

array([[0.2260416 , 0.8747859 , 0.44361995],
       [0.94700856, 0.96826017, 0.98426562]])

In [4]:
mask = spans[0].mask(0)
mask

array([[[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]]])

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Initialize the account first.
service = QiskitRuntimeService()
# Use `limit` to retrieve a specific number of jobs. The default `limit` is 10.
service.jobs(backend_name=my_backend)

## Cancel a job

You can cancel a job from the IBM Quantum Platform dashboard either on the Workloads page or the details page for a specific workload. On the Workloads page, click the overflow menu at the end of the row for that workload, and select Cancel. If you are on the details page for a specific workload, use the Actions dropdown at the top of the page, and select Cancel.

In Qiskit, use `job.cancel()` to cancel a job.

## Next steps

<Admonition type="tip" title="Recommendations">
    - Try the [Grover's algorithm](/docs/tutorials/grovers-algorithm) tutorial.
    - Learn more about [Sampler execution spans](/docs/guides/sampler-input-output#execution-spans)
</Admonition>